# Outliers with the percentile method
Detecting outliers in `Height` with the 1st and 99th percentiles, then trimming and capping (winsorization).

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split

## Load the data
10,000 people (5,000 men, 5,000 women): gender, height in inches, weight in pounds.

In [ ]:
df = pd.read_csv("data/weight-height.csv")
print(df.shape)
df.head()

In [ ]:
# the summary numbers of Height
df["Height"].describe()

## Look at the column

In [ ]:
# histogram above, box plot below; the dots beyond the whiskers are the box plot's outliers
fig = make_subplots(2, 1, shared_xaxes=True, row_heights=[0.7, 0.3])
fig.add_trace(go.Histogram(x=df["Height"]), 1, 1)
fig.add_trace(go.Box(x=df["Height"], quartilemethod="linear", name=""), 2, 1)
fig.update_layout(template="simple_white", showlegend=False, height=400)
fig.show()

## The limits: 1st and 99th percentiles

In [ ]:
# quantile takes a fraction: 0.99 is the 99th percentile
upper_limit = df["Height"].quantile(0.99)
lower_limit = df["Height"].quantile(0.01)
print("upper limit", upper_limit)
print("lower limit", lower_limit)

In [ ]:
# how many heights fall outside each limit
print("above", (df["Height"] > upper_limit).sum())
print("below", (df["Height"] < lower_limit).sum())

## Trimming

In [ ]:
# keep the rows between the two limits (use the variables, not rounded numbers)
new_df = df[df["Height"].between(lower_limit, upper_limit)]
print(new_df.shape)
new_df["Height"].describe()

In [ ]:
# typing rounded limits by hand loses one row: 74.785714 lies between 74.78 and 74.7858
rounded = df[(df["Height"] <= 74.78) & (df["Height"] >= 58.13)]
print(rounded.shape)

## Capping (winsorization)

In [ ]:
capped = df.copy()
# np.where(condition, value if true, value if false), nested twice:
# above the upper limit -> upper limit; below the lower limit -> lower limit; else unchanged
capped["Height"] = np.where(
    capped["Height"] > upper_limit,
    upper_limit,
    np.where(capped["Height"] < lower_limit,
             lower_limit,
             capped["Height"]),
)
print(capped.shape)
capped["Height"].describe()

In [ ]:
# the same thing in one line with clip
same = df["Height"].clip(lower_limit, upper_limit)
(same == capped["Height"]).all()

In [ ]:
# scipy has a ready-made winsorize: limits=(0.01, 0.01) cuts 1% at each end
from scipy.stats.mstats import winsorize
w = np.asarray(winsorize(df["Height"].to_numpy(), limits=(0.01, 0.01)))
# it caps at the most extreme kept value, not at the interpolated percentile
print(w.min(), w.max())

## Comparing: original, trimmed, capped

In [ ]:
versions = {"original": df, "trimmed": new_df, "capped": capped}
fig = make_subplots(3, 2, shared_xaxes=True, subplot_titles=[t for k in versions for t in (k, "")])
for i, part in enumerate(versions.values(), start=1):
    h = part["Height"]
    fig.add_trace(go.Histogram(x=h, xbins=dict(start=54, end=79.5, size=0.5)), i, 1)
    fig.add_trace(go.Box(x=h, quartilemethod="linear", name=""), i, 2)
fig.update_layout(template="simple_white", showlegend=False, height=700)
fig.show()

In [ ]:
# the summary numbers side by side
pd.DataFrame({k: v["Height"].describe() for k, v in versions.items()}).round(2)

## Try other cut-offs
The cut-offs are a choice. Each pair always flags about the same share of rows.

In [ ]:
for low, high in [(0.005, 0.995), (0.01, 0.99), (0.025, 0.975), (0.05, 0.95)]:
    a, b = df["Height"].quantile([low, high])
    n = ((df["Height"] < a) | (df["Height"] > b)).sum()
    print(f"{low:.3f}-{high:.3f}: limits {a:.2f} and {b:.2f}, {n} outliers")

## The three rules on the same column

In [ ]:
h = df["Height"]
mean, std = h.mean(), h.std()
q1, q3 = h.quantile([0.25, 0.75])
rules = {
    "z-score": (mean - 3 * std, mean + 3 * std),
    "IQR": (q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)),
    "percentile": (lower_limit, upper_limit),
}
for name, (a, b) in rules.items():
    print(f"{name:10s} {a:.2f} {b:.2f}  outliers: {((h < a) | (h > b)).sum()}")

## Better practice: learn the limits on the training set
The percentiles are learned from data, like a scaler's mean. We split first, learn them on the training rows, and apply the same limits to the test rows.

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
# limits from the training set only
lo, hi = train["Height"].quantile([0.01, 0.99])
print("training limits", lo, hi)

In [ ]:
# outliers in each part, judged with the training limits
for name, part in [("train", train), ("test", test)]:
    x = part["Height"]
    print(name, len(part), "rows:", (x < lo).sum(), "below,", (x > hi).sum(), "above")

In [ ]:
# trim the training set only; cap both parts with the training limits
train_trimmed = train[train["Height"].between(lo, hi)]
train_capped = train.assign(Height=train["Height"].clip(lo, hi))
test_capped = test.assign(Height=test["Height"].clip(lo, hi))
print(train_trimmed.shape, test_capped["Height"].min(), test_capped["Height"].max())

In [ ]:
# Weight: how many values lie beyond the box-plot fences, and how many the 1st/99th percentiles flag
w = df["Weight"]
q1, q3 = w.quantile([0.25, 0.75])
print("beyond IQR fences", ((w < q1 - 1.5 * (q3 - q1)) | (w > q3 + 1.5 * (q3 - q1))).sum())
a, b = w.quantile([0.01, 0.99])
print("beyond 1st/99th percentiles", ((w < a) | (w > b)).sum())